# cdfi-val — CDFI/MDI Valuation Toolkit
## Demo: Valuing Broadway Financial Corporation (BYFC)

Broadway Financial Corporation is one of the largest Black-owned banks
in the United States, headquartered in Los Angeles, CA. It is a certified
CDFI and MDI focused on affordable housing and small business lending
in underserved communities.

This notebook walks through a full valuation using TBV, P/E, and DCF.


In [ ]:
import sys
sys.path.insert(0, '..')

from cdfival import BankProfile, tbv, pe, dcf
from cdfival.portfolio.tracker import ValuationTracker
import pandas as pd
pd.set_option('display.max_columns', None)


## 1. Define the Institution

In [ ]:
broadway = BankProfile(
    name="Broadway Financial Corporation",
    ticker="BYFC",
    total_assets=655_000_000,
    tangible_common_equity=48_000_000,
    shares_outstanding=142_500_000,
    net_income_ltm=1_950_000,
    eps_ltm=0.014,
    cet1_ratio=0.122,
    institution_type="CDFI/MDI",
    fiscal_year_end="2023-12-31",
)

print(broadway)
print(f"Total Assets:  \${broadway.assets_mm:.1f}MM")
print(f"TCE:           \${broadway.tce_mm:.1f}MM")
print(f"CET1 Ratio:    {broadway.cet1_ratio*100:.1f}%")


## 2. Tangible Book Value (TBV) Valuation

TBV is the most widely used valuation method for community banks.
Peer MDI transactions have historically traded at 0.5x–0.9x TBV.


In [ ]:
tbv_result = tbv.value(
    broadway,
    multiple_range=(0.5, 0.9),
    steps=5,
    peer_median=0.70,
)
tbv_df = tbv_result.summary()


## 3. Price / Earnings (P/E) Valuation

P/E analysis using community bank peer multiples.
Broadway has thin but positive earnings — typical for MDIs of this size.


In [ ]:
pe_result = pe.value(
    broadway,
    multiple_range=(8.0, 16.0),
    steps=5,
    peer_median=11.0,
)
pe_df = pe_result.summary()


## 4. Discounted Cash Flow (DCF) Valuation

5-year projection with conservative 4% earnings growth,
discounted at 10% WACC with 2.5% terminal growth.


In [ ]:
dcf_result = dcf.value(
    broadway,
    wacc=0.10,
    terminal_growth=0.025,
    projection_years=5,
    earnings_growth_rate=0.04,
)
dcf_df = dcf_result.summary()


## 5. Portfolio Mode — Peer Comparison

Compare Broadway against another CDFI peer institution.


In [ ]:
unity = BankProfile(
    name="Unity Bancorp (Community)",
    ticker="UNTY",
    total_assets=520_000_000,
    tangible_common_equity=41_000_000,
    shares_outstanding=95_000_000,
    net_income_ltm=1_700_000,
    eps_ltm=0.018,
    cet1_ratio=0.115,
    institution_type="CDFI",
)

tracker = ValuationTracker()
tracker.add(broadway)
tracker.add(unity)

print("=== TBV Portfolio Summary ===")
tbv_table = tracker.summary_table(method="tbv", multiple_range=(0.5, 0.9), steps=3)
print(tbv_table.to_string(index=False))

print("\n=== DCF Portfolio Summary ===")
dcf_table = tracker.summary_table(method="dcf", wacc=0.10, terminal_growth=0.025)
print(dcf_table.to_string(index=False))


## Summary

| Method | Key Input | Implied Price Range |
|--------|-----------|-------------------|
| TBV | 0.5x – 0.9x P/TBV | See output above |
| P/E | 8x – 16x | See output above |
| DCF | WACC 10%, g 2.5% | See output above |

**GitHub:** https://github.com/Jaypatel1511/cdfi-val  
**PyPI:** https://pypi.org/project/cdfi-val
